## Viken KHATCHERIAN
## Formation AI Engineer
## Openclassrooms
## Projet 6 : initiez-vous au MLOps partie (1/2)
## Notebook 2 : 02_tracking_exp_mlflow_p6_vk.ipynb
### ETAPE 2 : TRACKING DES EXPERIMENTATIONS AVEC MLFLOW

In [1]:
# Importer les bibliothèques
import mlflow
import mlflow.sklearn
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

/mnt/projects/ai_engineer_projects/P06/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Configuration de mlflow
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("P06_baselines")

<Experiment: artifact_location='file:///mnt/projects/ai_engineer_projects/P06/mlruns/5', creation_time=1780649542921, experiment_id='5', last_update_time=1780649542921, lifecycle_stage='active', name='P06_baselines', tags={}, trace_location=None, workspace='default'>

In [3]:
# Chargement des données
X_train = pd.read_parquet("X_train_ml.parquet")
y_train = pd.read_parquet("y_train_ml.parquet").squeeze()

print("Shape X_train :", X_train.shape)
print("Shape y_train :", y_train.shape)

Shape X_train : (307511, 250)
Shape y_train : (307511,)


In [4]:
# Split train/validation
X_tr, X_val, y_tr, y_val = train_test_split(
X_train,
y_train,
test_size=0.20,
random_state=42,
stratify=y_train
)

In [5]:
# Run 1 : logistic regression baseline
with mlflow.start_run(run_name="logreg_baseline"):

    model = LogisticRegression(
        max_iter=5000,
        class_weight="balanced",
        n_jobs=8
    )
    
    model.fit(X_tr, y_tr)
    
    preds = model.predict_proba(X_val)[:, 1]
    
    auc = roc_auc_score(y_val, preds)
    
    # Tags
    mlflow.set_tags({
        "project": "P06",
        "stage": "baseline",
        "model_family": "logistic_regression"
    })

    mlflow.set_tag(
    "mlflow.note.content",
    """
    Premier essai de tracking mlflow avec une régression logistique sans normalisation des features.

    Ce premier run porte le nom de logreg_baseline.
    """
    )
    
    # Paramètres
    mlflow.log_param("dataset", "X_train_ml.parquet")
    mlflow.log_param("n_features", X_train.shape[1])
    
    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_param("max_iter", 5000)
    mlflow.log_param("class_weight", "balanced")
    
    mlflow.log_param("test_size", 0.20)
    mlflow.log_param("random_state", 42)
    
    # Métriques
    mlflow.log_metric("auc", auc)

    # Enregistrer le modèle dans un Modèle Registry
    mlflow.sklearn.log_model(
    model,
    artifact_path="model",
    registered_model_name="P06_LogReg_Baseline"
    )

print(f"Baseline AUC : {auc:.4f}")

auc_baseline = auc

/mnt/projects/ai_engineer_projects/P06/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=8', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/mnt/projects/ai_engineer_projects/P06/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 5000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=5000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
2026/06/06 14:43:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please us

🏃 View run logreg_baseline at: http://127.0.0.1:5000/#/experiments/5/runs/399b13c37c274699a93188a917d010b5
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5
Baseline AUC : 0.6562


In [8]:
# Run 2 : logistic regression + standard scaler + modèle documenté dans Model Registry de mlflow
from mlflow import MlflowClient

with mlflow.start_run(run_name="logreg_scaled") as run:

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(
            max_iter=5000,
            class_weight="balanced",
            n_jobs=8
        ))
    ])
    
    model.fit(X_tr, y_tr)
    
    preds = model.predict_proba(X_val)[:, 1]
    
    auc = roc_auc_score(y_val, preds)
    
    # Tags
    mlflow.set_tags({
        "project": "P06",
        "stage": "baseline",
        "model_family": "logistic_regression"
    })

    mlflow.set_tag(
    "mlflow.note.content",
    """
    Second essai de tracking mlflow avec une régression logistique avec normalisation des features.

    Ce second run porte le nom de logreg_scaled.
    """
    )
    
    # Paramètres
    mlflow.log_param("dataset", "X_train_ml.parquet")
    mlflow.log_param("n_features", X_train.shape[1])
    
    mlflow.log_param("model", "LogisticRegression")
    
    mlflow.log_param("scaler", "StandardScaler")
    mlflow.log_param("max_iter", 5000)
    mlflow.log_param("class_weight", "balanced")
    
    mlflow.log_param("test_size", 0.20)
    mlflow.log_param("random_state", 42)
    
    # Métriques
    mlflow.log_metric("auc", auc)
    
    # Enregistrer le modèle dans un Modèle Registry
    result = mlflow.sklearn.log_model(
        model,
        artifact_path="model",
        registered_model_name="P06_LogReg_Baseline_scaled"
        )

    run_id = run.info.run_id

    # version du modèle dans registry
    model_version = result.registered_model_version

# métadonnées du modèle registry
client = MlflowClient()

client.update_model_version(
    name="P06_LogReg_Baseline_scaled",
    version=model_version,
    description="LogReg baseline balanced class_weight,scaled features model P06"
)

client.set_model_version_tag(
    name="P06_LogReg_Baseline_scaled",
    version=model_version,
    key="stage",
    value="baseline"
)

client.set_model_version_tag(
    name="P06_LogReg_Baseline_scaled",
    version=model_version,
    key="project",
    value="P06"
)

client.set_registered_model_alias(
    name="P06_LogReg_Baseline_scaled",
    alias="lr_base_scaled",
    version=model_version
)

print(f"Scaled AUC : {auc:.4f}")

print("\nTracking MLflow terminé.")

/mnt/projects/ai_engineer_projects/P06/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=8', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
2026/06/06 15:04:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/06 15:04:09 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/06 15:04:10 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Registered model 'P06_Log

🏃 View run logreg_scaled at: http://127.0.0.1:5000/#/experiments/5/runs/15cce76d78d2419ca1664a983f9d403c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5
Scaled AUC : 0.7684

Tracking MLflow terminé.
